In [ ]:
import pandas as pd
from glob import glob
import os
import matplotlib.pyplot as plt
from scipy import ndimage
import skimage as sk
import numpy as np
from PIL import Image
import math
import napari

In [ ]:
def open_img(
        img_path: str
    ):
    arr = np.array(Image.open(img_path).convert('L'))
    return arr

# def get_tissue_bbox(
#         tissue_arr,
#         name,
#         tissue_id,
#         data_path
#     ):
#     # mask = ndimage.binary_fill_holes(mask)
#     # labels = sk.measure.label(mask)
#     props = sk.measure.regionprops(tissue_arr)
#     largest_obj = max(props, key=lambda p: p.area)
#     largest_obj_label = largest_obj.label
#     minc = min(props, key=lambda p: p.bbox[0])
#     minr = min(props, key=lambda p: p.bbox[1])
#     maxc = max(props, key=lambda p: p.bbox[2])
#     maxr = max(props, key=lambda p: p.bbox[3])
#     img_bbox = tissue_arr[minc.bbox[0]-10:maxc.bbox[2]+10,minr.bbox[1]-10:maxr.bbox[3]+10]
#     cropped_mask = (img_bbox==largest_obj_label)*img_bbox
#     cropped_mask = ndimage.binary_fill_holes(cropped_mask)
#     cropped_mask = sk.img_as_ubyte(cropped_mask)
#     # save_path_img = os.path.join(data_path,tissue_id,'cropped_image')
#     # save_path_mask = os.path.join(data_path,tissue_id,'cropped_mask')
#     # os.makedirs(save_path_img,exist_ok=True)
#     # os.makedirs(save_path_mask,exist_ok=True)
#     # sk.io.imsave(os.path.join(save_path_img,'cropped_'+name+'.png'),img_bbox,check_contrast=False)
#     # sk.io.imsave(os.path.join(save_path_mask,'cropped_mask_'+name+'.png'),cropped_mask,check_contrast=False)
#     return img_bbox, cropped_mask

# def get_map_region(
#         map_arr,
#         map_id,
#         grey_value):
#     #grey_value = grey_value_df.loc[grey_value_df['Mapping_ID'] == map_id, 'Map_Grey_value'].values[0] 
#     map_region = (map_arr == grey_value).astype(np.float32)
#     map_region = ndimage.binary_fill_holes(map_region)
#     labels = sk.measure.label(map_region)
#     props = sk.measure.regionprops(labels)
#     largest_obj = max(props, key=lambda p: p.area)
#     largest_obj_label = largest_obj.label
#     masked_map_region = (labels==largest_obj_label)*map_region
#     if not map_region.any():
#         raise ValueError(f"Map region for ID '{map_id}' (grey value {grey_value}) is empty check map image and grey value key.")
#     return masked_map_region

In [ ]:
def add_padding(
            img_bbox,
            cropped_mask,
            name,
            tissue_id,
            data_path):
        """ 
        reshape the cropped images to a square based on the largest dim using padding
        saves new arrays with added padding of a constant value: 255 for grey scale image, 0 for masks
        returns save path for later use
        """
        max_dim = max(img_bbox.shape)
        padding_y = (max_dim - img_bbox.shape[0]) // 2
        padding_x = (max_dim - img_bbox.shape[1]) // 2
        pad_img_bbox = np.pad(img_bbox,((padding_y+50,padding_y+50),(padding_x+50,padding_x+50)),mode='constant',constant_values=0)
        pad_mask_bbox = np.pad(cropped_mask,((padding_y+50,padding_y+50),(padding_x+50,padding_x+50)),mode='constant',constant_values=0)
        # save_path_img = os.path.join(data_path,tissue_id,'padded_cropped_image')
        # save_path_mask = os.path.join(data_path,tissue_id,'padded_cropped_mask')
        # os.makedirs(save_path_img,exist_ok=True)
        # os.makedirs(save_path_mask,exist_ok=True)
        # sk.io.imsave(os.path.join(save_path_img,'padded_cropped_'+name+'.png'),pad_img_bbox,check_contrast=False)
        # sk.io.imsave(os.path.join(save_path_mask,'padded_cropped_mask_'+name+'.png'),pad_mask_bbox,check_contrast=False)
        return pad_img_bbox,pad_mask_bbox

In [ ]:
tissue = open_img(img_path=r"\\pn.vai.org\projects\steensma\vari-core-generated-data\Involution_Atlas_collab\KG_Registration_Pipeline\Masks_For_Reg_Angle\Tissue2\192281.svs.png")
map = open_img(img_path=r"\\pn.vai.org\projects\steensma\vari-core-generated-data\Involution_Atlas_collab\KG_Registration_Pipeline\GrayscaleMaps\resized_maps\SD183_Left.png")

In [ ]:
north_ambig_ex = open_img(r"\\pn.vai.org\projects\steensma\vari-core-generated-data\Involution_Atlas_collab\KG_Registration_Pipeline\Results_Registration_Angles_V4\Tissue3\padded_cropped_image\padded_cropped_192279.png")

In [ ]:
viewer = napari.view_labels(north_ambig_ex)

In [65]:
def detect_cardinal_point(
        pad_img_bbox, 
        grey_value, 
        name):
    """
    Detect the centroid of a cardinal point marker by its grey value.
    
    Returns (x, y) in pixel coordinates, or None if not found.
    """
    
    # Create binary mask for this grey value.
    # Use a small tolerance instead of exact equality to avoid failures when images
    # were saved with lossy settings or resampled previously.
    gv = grey_value
    mask = (pad_img_bbox==gv)
        
    if not mask.any():
        print(f"  WARNING: No pixels found for grey value {grey_value} in {name}")
        return None
    
    # Label connected components and get centroid
    labeled_img = sk.measure.label(mask.astype(np.uint8))
    props = sk.measure.regionprops(labeled_img)
    if not props:
        print(f"  WARNING: No connected components found for grey value {grey_value} in {name}")
        return None

    largest = max(props, key=lambda p: p.area)
    cy, cx = largest.centroid
    return np.array([cy, cx], dtype=float)  # return as (y,x)

def get_cardinal_points(
        pad_img_bbox,
        name,
        grey_value_df,
        tissue,
        ):
    """
    Extract one or more cardinal points from an image.

    Parameters
    ----------
    direction:
        Either a single direction string (e.g., 'north') or an iterable of
        directions (e.g., ('north', 'east')).

    Returns
    -------
    dict[str, np.ndarray]
        Keys are directions and values are (y,x) arrays (float).
    """
    points = {'north':[],
                'east':[]}
    keys = ['north','east']
    if tissue == 'Tissue':
        directions = ('North','East')
    if tissue == 'Tissue1':
        directions = ('North1','East1')
    if tissue == 'Tissue2':
        directions = ('North2','East2')
    if tissue == 'Tissue3':
        directions = ('North3','East3')
    for i,d in enumerate(directions):
        grey = grey_value_df.loc[grey_value_df['Mask_IDs'] == d, 'Mask_Grey_Value'].values[0]
        print(f'Grey value for {d} is {grey}')
        pt = detect_cardinal_point(pad_img_bbox, grey, name)
        if pt is None:
            raise ValueError(f"Could not find '{d}' point in {name}")
        points[keys[i]] = pt
        print(f"  {d}: pixel (y,x) ({pt[0]:.1f}, {pt[1]:.1f})")
    return points

In [ ]:
grey_value_df = pd.read_csv(r"E:\GitHub_Repos\Mammary_Gland_Involution\Map_Registration\Maps\PNGsForReg\HexColorMappingKey.csv",dtype=str)
grey_value_df['Mask_Grey_Value'] = grey_value_df['Mask_Grey_Value'].astype(float)

In [ ]:
points = get_cardinal_points(north_ambig_ex,'test_points',grey_value_df,'Tissue3')

In [ ]:
viewer.add_points(points['north'],name='North')
viewer.add_points(points['east'],name='East')

Don't like the calculation, seems overly complicated because the angle is opposite of what needs to occur (which is corrected for in the registration)

In [ ]:
def AI_orient_tissue(
            points,
            name,
            pad_img):
        """
        Determine the flip and/or rotation needed to orient the moving image so that
        north points up and east points left, with "true north" defined as orthogonal
        to the detected east-west axis.

        Parameters
        ----------
        points:
            dict with 'north' and 'east' as (y,x) pixel arrays in the moving image.
        mask_arr:
            Tissue mask array used to compute the tissue centroid (non-zero pixels treated as tissue).
        """
        if 'north' not in points or 'east' not in points:
            raise ValueError(f"Missing required cardinal points for {name}. Found: {list(points.keys())}")
        img_centroid = (pad_img.shape[0]//2,pad_img.shape[1]//2)
        print(f'image centroid: {img_centroid}')
        tissue_pixels = np.argwhere(pad_img == 3)
        print(f'tissue_pixels array: {tissue_pixels.shape}')
        print(f'input array: {pad_img.shape}')
        if tissue_pixels.size == 0:
            raise ValueError(f"No tissue mask pixels found for {name}")
        cy, cx = tissue_pixels.mean(axis=0)  # (y, x)
        print(f'mask centroid: (y,x) {cy},{cx}')

        def _wrap_degrees(deg: float) -> float:
            # Map to [-180, 180)
            wrap_deg = ((deg + 180.0) % 360.0) - 180.0
            print(f'wrap deg: {wrap_deg}')
            return wrap_deg

        # Work in vector form around the tissue centroid to avoid dependence on image center.
        # Use EAST to define the true east-west axis, then choose the 180° branch that makes NORTH point up.
        y_n, x_n = points['north']
        y_e, x_e = points['east']
        dx_n, dy_n = (x_n - cx), (y_n - cy)
        dx_e, dy_e = (x_e - cx), (y_e - cy)
        print(f'vector north (x,y): {dx_n}, {dy_n}; vector east (x,y): {dx_e}, {dy_e}')
        viewer.add_points([dx_n, dy_n], name='vector north')
        viewer.add_points([dx_e, dy_e],name='vector east')

        # CCW rotation (screen/image coordinates) needed to rotate the EAST vector to point LEFT (-x).
        # angle_e is the current angle of EAST relative to +x; desired is 180° (left).
        angle_e = math.degrees(math.atan2(dy_e, dx_e))
        print(f'angle_e: {angle_e}')
        angle_degrees = _wrap_degrees(180.0 - angle_e)
        print(f'angle_degrees: {angle_degrees}')

        def _rotate(dx: float, dy: float, deg: float) -> tuple[float, float]:
            r = math.radians(deg)
            c = math.cos(r)
            s = math.sin(r)
            rotate = (c * dx - s * dy, s * dx + c * dy)
            print(f'rotate formula: {(c * dx - s * dy, s * dx + c * dy)}')
            return rotate

        # Ensure NORTH ends up pointing up (-y). If not, rotate an additional 180°.
        _, dy_n_rot = _rotate(dx_n, dy_n, angle_degrees)
        if dy_n_rot > 0:
            angle_degrees = _wrap_degrees(angle_degrees + 180.0)

        # Determine horizontal flip after rotation to enforce EAST to the left.
        dx_e_rot, _ = _rotate(dx_e, dy_e, angle_degrees)
        if abs(dx_e_rot) < 1.0:
            print('East is on (or too close to) the midline after rotation, check image')
            raise ValueError("ORIENTATION_MISMATCH")
        flip = 'horizontal' if dx_e_rot > 0 else None

        rads = math.radians(angle_degrees)
        print(f'Detected Flip: {flip}')
        print(f'Detected Rotation: {angle_degrees:.2f} degrees ({rads:.4f} radians)')
        return flip, rads, angle_degrees

In [ ]:
def KG_orient_tissue(
        points,
        name,
        grey_value_df,
        mask_arr):
    """
    Determine the flip and/or rotation needed to orient the moving image
    to match the map, using the N and E cardinal point locations.

    Calculate the angle of rotation to orient north up and then determine if a horizontal flip
    is needed to place East to the left.

    Parameters
    ----------
    mask_points  : dict with 'north' and 'east' as (x, y) pixel arrays — moving image
    moving_image: bbox image np.array of moving image with cardinal points
    Returns
    -------
    rotation angle
    flip matrix
    """
    # --- Detect required rotation from cardinal point geometry ---
    # Compare pixel coords directly for orientation — spacing does not
    # affect which direction is "up" or "right"

    #calculate angle between relative north and cardinal north coordinate:
    north_relative = (mask_arr.shape[0]//2,0)
    print(f'relative North: {north_relative}') #relative north
    north_cardinal = points['north']
    print(f'cardinal north: {north_cardinal}')

    y1, x1 = north_relative
    x2, y2 = north_cardinal

    dx = x2 - x1
    dy = y2 - y1
    print(f'distance x: {dx}, distance y: {dy}')

    angle_radians = math.atan2(dy, dx)
    angle_degrees = math.degrees(angle_radians)
    
    #rotate image to place north up, relocate East and determine if horizontal flip is needed
    #rotated_img = sk.transform.rotate(mask_arr,angle=angle_degrees,preserve_range=True,mode='constant')
    rotated_img = ndimage.rotate(mask_arr,angle=angle_degrees)
    viewer.add_image(rotated_img, name='KG rotation function')
    rot_points = get_cardinal_points(rotated_img,name,grey_value_df,tissue='Tissue3')
    rot_east = rot_points['east']
    print(f'new east: {rot_east}')

    mid_point_y = rotated_img.shape[0]//2
    print(f'mid point y: {mid_point_y}')
    _,east_y = rot_east

    if east_y > mid_point_y:
        flip = 'horizontal'
    elif east_y < mid_point_y:
        flip = None
    else:
        print('East is on the midline, check image')
        flip = None
        raise ValueError(f"ORIENTATION_MISMATCH")
    print(f'Detected Flip: {flip}')
    print(f'Detected Rotation(radians): {angle_radians}')
    return flip, angle_radians, angle_degrees

In [ ]:
def spliced_function(points,
        name,
        grey_value_df,
        mask_arr,
        tissue):
    
    if 'north' not in points or 'east' not in points:
        raise ValueError(f"Missing required cardinal points for {name}. Found: {list(points.keys())}")
    tissue_pixels = np.argwhere(mask_arr == 3)
    if tissue_pixels.size == 0:
        raise ValueError(f"No tissue mask pixels found for {name}")
    #Use centroid of mask to calculate angle of rotation
    cy, cx = tissue_pixels.mean(axis=0)  # (y, x)
    centroid = (cy,cx)
    #cy, cx = (mask_arr.shape[0]//2,mask_arr.shape[1]//2)
    y_n, x_n = points['north'] # <- my brain does not understand why the north point is not required for this calculation
    #y_e, x_e = points['east'] #coordinate of east cardinal point
    dx_n, dy_n = (x_n - cx), (y_n - cy)
    #quadrant correction standard:
    # if dx_n > 0:
    #     if dy_n > 0:
    #         print(f'Points in QI')
    #         angle = (np.pi/2) - math.degrees(math.atan2(dy_n, dx_n))
    #     elif dy_n < 0:
    #         print(f'Points in QIV')
    #         angle = math.degrees(math.atan2(dy_n, dx_n)) - ((3*np.pi)/2)
    # elif dx_n < 0:
    #     if dy_n > 0:
    #         print(f'Points in QII')
    #         angle = math.degrees(math.atan2(dy_n, dx_n)) - (np.pi/2)
    #     elif dy_n < 0:
    #         print(f'Points in QIII')
    #         angle = ((3*np.pi)/2) - math.degrees(math.atan2(dy_n, dx_n))

    #quadrant correction opposite:
    if dx_n > 0:
        if dy_n > 0:
            print(f'Points in QI')
            angle = math.degrees(math.atan2(dy_n, dx_n)) - (np.pi/2) 
        elif dy_n < 0:
            print(f'Points in QIV')
            angle = ((3*np.pi)/2) - math.degrees(math.atan2(dy_n, dx_n))   
    elif dx_n < 0:
        if dy_n > 0:
            print(f'Points in QII')
            angle = (np.pi/2) - math.degrees(math.atan2(dy_n, dx_n))
        elif dy_n < 0:
            print(f'Points in QIII')
            angle = math.degrees(math.atan2(dy_n, dx_n)) - ((3*np.pi)/2)  


    print(f'adjusted north coords (x,y): {dx_n},{dy_n}')
    print(f'rotation angle: {angle}')
    angle_radians = np.deg2rad(angle)
    flip = []
    #rotated_img = ndimage.rotate(mask_arr,angle=angle_adj,order=0,)
    #viewer.add_labels(rotated_img, name='Spliced rotation function')
    # rot_points = get_cardinal_points(rotated_img,name,grey_value_df,tissue=tissue)
    # rot_east = rot_points['east']
    # print(f'new east: {rot_east}')
    # mid_point_x = rotated_img.shape[1]//2
    # print(f'mid point y: {mid_point_x}')
    # east_x,_ = rot_east
    # if east_x > mid_point_x:
    #     flip = 'horizontal'
    # elif east_x < mid_point_x:
    #     flip = None
    # else:
    #     print('East is on the midline, check image')
    #     flip = None
    #     raise ValueError(f"ORIENTATION_MISMATCH")
    # print(f'Detected Flip: {flip}')
    print(f'Detected Rotation(radians): {angle_radians}')
    return flip, angle_radians, angle, centroid
    
    

In [ ]:
test = open_img(r"\\pn.vai.org\projects\steensma\vari-core-generated-data\Involution_Atlas_collab\KG_Registration_Pipeline\Results_Registration_Angles_V4\Tissue\padded_cropped_image\padded_cropped_228873.png")
points = get_cardinal_points(test, name=None,grey_value_df=grey_value_df,tissue='Tissue')
viewer = napari.view_labels(test,name='original orientation')
viewer.add_points(points['east'],name='East',face_color='Magenta')
viewer.add_points(points['north'],name='North')

In [ ]:
_, angle_radians, angle_degrees, mask_centroid = spliced_function(points,'rotation test',grey_value_df=None,mask_arr=test,tissue=None)

In [98]:
def rotate_array_around_center(arr, angle_degrees, centroid, reshape=True,order=0,mode='nearest',prefilter=False):
    """
    Rotates an array around a specific center coordinate.

    Args:
        points (np.ndarray): An N x 2 array of (x, y) coordinates.
        angle_degrees (float): The rotation angle in degrees (counter-clockwise).
        center (tuple): The (x0, y0) coordinates of the rotation center.

    Returns:
        np.ndarray: The array of rotated array.
    """
    # Convert angle to radians
    c_y,c_x = arr.shape[0]//2,arr.shape[1]//2
    t_y,t_x = centroid
    s_y,s_x = c_y - t_y, c_x - t_x

    #Shift Array to new center:
    shifted_arr = ndimage.shift(arr, [s_y,s_x], mode=mode,prefilter=prefilter,order=order)

    #Rotate array around new point:
    rotated_arr = ndimage.rotate(shifted_arr, angle_degrees, reshape=reshape,order=order, mode=mode, prefilter=prefilter)
        
    return rotated_arr

In [ ]:
new_array = rotate_array_around_center(test, angle_degrees, mask_centroid)
viewer.add_labels(new_array,name='rotated 7')

In [94]:
#Try using three points for angle:
def calculate_angle_and_flip(points,
        name,
        grey_value_df,
        mask_arr,
        tissue):
    """
    Calculates the angle (in degrees) between three points.

    north coord : [x, y]
    mask centroid : [x, y] (vertex)
    relative north over mask centroid  : [x, y]
    """
    ny, nx = points['north']
    north_coord = (nx, ny)

    tissue_pixels = np.argwhere(mask_arr == 3)
    #Use centroid of mask to calculate angle of rotation
    cy, cx = tissue_pixels.mean(axis=0)  # (y, x)
    mask_centroid = (cx,cy)
    #Use north over mask centroid
    relative_north = (cx, 0)

    #Get side of image north is on
    if nx < mask_arr.shape[1]//2:
        side = 'left'
    elif nx < mask_arr.shape[1]//2:
        side = 'right'
    else:
        side = 'midline'

    #if north on mid line, determine if on the top or bottom of image:
    if side == 'midline':
        if ny < mask_arr.shape[0]//2:
            rotate = None
        if ny > mask_arr.shape[0]//2:
            rotate = 180
    # Convert points to numpy arrays
    p1 = np.array(north_coord)
    p2 = np.array(mask_centroid)
    p3 = np.array(relative_north)

    # Create vectors from the vertex (p2) to the other points
    v1 = p1 - p2
    v2 = p3 - p2

    # Calculate the dot product and magnitudes
    dot_product = np.dot(v1, v2)
    magnitude_v1 = np.linalg.norm(v1)
    magnitude_v2 = np.linalg.norm(v2)

    # Calculate the cosine of the angle
    # Ensure the denominator is not zero to avoid division errors
    if magnitude_v1 == 0 or magnitude_v2 == 0:
        return 0
    cosine_angle = dot_product / (magnitude_v1 * magnitude_v2)

    # Handle floating point errors that might result in a value slightly outside [-1, 1]
    cosine_angle = np.clip(cosine_angle, -1.0, 1.0)

    # Calculate the angle in radians and convert to degrees
    angle_rad = np.arccos(cosine_angle)
    angle_deg = np.degrees(angle_rad)

    if side == 'left':
        angle_deg = -angle_deg
        angle_rad = np.deg2rad(angle_deg)
    elif side == 'right':
        angle_deg = angle_deg
        angle_rad = np.deg2rad(angle_deg)
    elif rotate:
        angle_deg = rotate
        angle_rad = np.deg2rad(angle_rad)

    rotated_arr = rotate_array_around_center(mask_arr,angle_deg,centroid=(cy,cx))
    viewer.add_labels(rotated_arr, name='rotated')
    rotated_points = get_cardinal_points(rotated_arr,name,grey_value_df,tissue)

    new_east = rotated_points['east']
    print(f'New East Coords: {new_east}')
    print(f'New array shape: {rotated_arr.shape}')
    _, ex = new_east

    if ex > rotated_arr.shape[1]//2:
        flip = 'hortizontal'
    elif ex < rotated_arr.shape[1]//2:
        flip = None

    print(f'Detected Rotation(degrees): {angle_deg}')
    print(f'Detected Rotation(radians): {angle_rad}')
    print(f'Detected Flip: {flip}')
    return angle_deg, angle_rad, flip



In [ ]:
relative_north = (mask_centroid[1],0)

In [ ]:
angle = calculate_angle(points['north'],mask_centroid,relative_north)

In [ ]:
print(angle)

In [ ]:
rotated_arr = rotate_array_around_center(test,angle,mask_centroid)
viewer.add_labels(rotated_arr,name='basic angle calc with mask based north')

In [90]:
test = open_img(r"\\pn.vai.org\projects\steensma\vari-core-generated-data\Involution_Atlas_collab\KG_Registration_Pipeline\Results_Registration_Angles_V4\Tissue2\padded_cropped_image\padded_cropped_192277.png")
points = get_cardinal_points(test, name=None,grey_value_df=grey_value_df,tissue='Tissue1')
viewer = napari.view_labels(test,name='original orientation')
#viewer.add_points(points['east'],name='East',face_color='Magenta')
#viewer.add_points(points['north'],name='North')

Grey value for North1 is 1.0
  North1: pixel (y,x) (1014.3, 812.0)
Grey value for East1 is 2.0
  East1: pixel (y,x) (828.0, 80.2)


E:\Temp\ipykernel_52024\1967684127.py:3: FutureWarning: `napari.view_labels` is deprecated and will be removed in napari 0.7.0.
Use `viewer = napari.Viewer(); viewer.add_labels(...)` instead.
  viewer = napari.view_labels(test,name='original orientation')


In [69]:
grey_value_df = pd.read_csv(r"E:\GitHub_Repos\Mammary_Gland_Involution\Map_Registration\Maps\PNGsForReg\HexColorMappingKey.csv",dtype=str)
grey_value_df['Mask_Grey_Value'] = grey_value_df['Mask_Grey_Value'].astype(float)

In [99]:
angle_deg, angle_rad, flip = calculate_angle_and_flip(points,'test',grey_value_df,test,'Tissue1')

Grey value for North1 is 1.0
  North1: pixel (y,x) (526.7, 512.0)
Grey value for East1 is 2.0
  East1: pixel (y,x) (713.0, 1243.8)
New East Coords: [ 712.9787234  1243.82978723]
New array shape: (1501, 1501)
Detected Rotation(degrees): 180
Detected Rotation(radians): 0.040580199471793046
Detected Flip: hortizontal


In [96]:
1501//2

750

In [97]:
new_east = [926.5, 656.23809524]
viewer.add_points(new_east)

<Points layer 'Points' at 0x1fca7bb89d0>